In [17]:
import re
import emoji
import plotly.express as px
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "cardiffnlp/twitter-roberta-base-irony"
MODEL_REVISION = "3bf8f118bdf6b00c99658151ef10c9a0b9afd6bf"

In [18]:
tweets = load_dataset("cardiffnlp/tweet_eval", "irony", split="test").to_pandas()
tweets["reference_label"] = tweets["label"].map({
    0: "Non-ironic",
    1: "Ironic",
})

explicit_tag = re.compile(r"(?<!\w)#(?:irony|sarcasm|not)\b", re.IGNORECASE)

tweets["removed_tags"] = tweets["text"].map(
    lambda text: ", ".join(explicit_tag.findall(text))
)
tweets["edited_text"] = tweets["text"].map(
    lambda text: explicit_tag.sub("", text).strip()
)

tweets.groupby("reference_label")["removed_tags"].agg(
    total="size",
    with_removed_tags=lambda tags: tags.ne("").sum(),
)

,total,with_removed_tags
reference_label,,
Ironic,311,311
Non-ironic,473,113


In [19]:
# Resolve the Hub checkpoint explicitly, even if a local folder has the same name.
checkpoint = snapshot_download(
    MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=["config.json", "pytorch_model.bin", "merges.txt",
                    "vocab.json", "special_tokens_map.json"],
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.eval()


def normalize_tweet(text):
    tokens = []
    for token in text.split():
        if token.startswith("@") and len(token) > 1:
            token = "@user"
        elif token.startswith("http"):
            token = "http"
        tokens.append(token)
    return " ".join(tokens)


@torch.inference_mode()
def irony_scores(texts, batch_size=16):
    scores = []

    for start in range(0, len(texts), batch_size):
        batch = [
            normalize_tweet(text)
            for text in texts[start:start + batch_size]
        ]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        logits = model(**inputs).logits
        scores.extend(logits.softmax(dim=-1)[:, 1].cpu().tolist())

    return scores


tweets["score_before"] = irony_scores(tweets["text"].tolist())

changed = tweets.loc[tweets["removed_tags"].ne("")].copy()
changed["score_after"] = irony_scores(changed["edited_text"].tolist())
changed["score_delta"] = changed["score_after"] - changed["score_before"]

changed["class_flip"] = (
    changed["score_before"].ge(0.5) != changed["score_after"].ge(0.5)
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26885.71it/s]


In [20]:
if changed.empty:
    print("Im Testsplit wurden keine Hashtags gefunden.")
else:
    fig = px.scatter(
        changed,
        x="score_before",
        y="score_after",
        color="reference_label",
        symbol="class_flip",
        opacity=0.65,
        category_orders={
            "reference_label": ["Ironic", "Non-ironic"],
            "class_flip": [False, True],
        },
        hover_data={
            "text": True,
            "edited_text": True,
            "removed_tags": True,
            "score_delta": ":.3f",
        },
        labels={
            "score_before": "Ironie-Score · Original",
            "score_after": "Ironie-Score · ohne Ironie-Hashtags",
            "reference_label": "Referenzlabel",
            "class_flip": "Klassenwechsel",
            "text": "Originaltext",
            "edited_text": "Bearbeiteter Text",
            "removed_tags": "Entfernte Hashtags",
            "score_delta": "Score-Änderung",
        },
        template="plotly_white",
    )

    fig.add_shape(
        type="line",
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color="gray", dash="dash"),
    )
    fig.add_vline(x=0.5, line_dash="dot")
    fig.add_hline(y=0.5, line_dash="dot")
    fig.update_xaxes(range=[0, 1], tick0=0, dtick=0.2, constrain="domain")
    fig.update_yaxes(
        range=[0, 1],
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    )
    legend_labels = {
        "Ironic, False": "Ironisch",
        "Ironic, True": "Ironisch · Wechsel",
        "Non-ironic, False": "Nicht ironisch",
        "Non-ironic, True": "Nicht ironisch · Wechsel",
    }
    fig.for_each_trace(lambda trace: trace.update(name=legend_labels[trace.name]))
    fig.update_xaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_yaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_layout(
        width=600,
        modebar_remove=["toImage", "select2d", "lasso2d", "zoomIn2d",
                        "zoomOut2d", "autoScale2d"],
        height=640,
        margin=dict(l=61, r=19, t=66, b=54),
        title=dict(
            text=f"Hashtag-Entfernung · {len(changed)} Tweets",
            x=0,
            xref="paper",
            xanchor="left",
            y=0.985,
            yanchor="top",
            font=dict(size=21),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=1.012,
            yanchor="bottom",
            title_text="",
            font=dict(size=11),
        ),
    )
    fig.show()

In [21]:
histogram = px.histogram(
    tweets,
    x="score_before",
    facet_col="reference_label",
    histnorm="percent",
    nbins=20,
    labels={"score_before": "Ironie-Score", "reference_label": "Referenzlabel"},
    title="Ironie-Scores im gesamten Testsplit",
)
histogram.add_vline(x=0.5, line_dash="dot")
histogram.update_xaxes(range=[0, 1])
histogram.show()

# Für den Vergleich einzelner Tags nur Tweets mit genau einem Treffer verwenden.
single_tag = changed.loc[~changed["removed_tags"].str.contains(",")].copy()
single_tag["marker"] = single_tag["removed_tags"].str.lower()
single_tag["n"] = single_tag.groupby(
    ["marker", "reference_label"]
)["score_delta"].transform("size")

boxplot = px.box(
    single_tag,
    x="marker",
    y="score_delta",
    color="reference_label",
    hover_data=["n"],
    labels={
        "marker": "Entfernter Hashtag",
        "score_delta": "Ironie-Score nachher minus vorher",
        "reference_label": "Referenzlabel",
    },
    title="Score-Änderung bei Tweets mit genau einem Hashtag-Treffer",
)
boxplot.add_hline(y=0, line_dash="dot")
boxplot.show()

changed.loc[
    changed["score_before"].ge(0.5) & changed["score_after"].lt(0.5),
    [
        "text", "edited_text", "removed_tags", "reference_label",
        "score_before", "score_after", "score_delta",
    ],
].sort_values("score_delta").head(10)

,text,edited_text,removed_tags,reference_label,score_before,score_after,score_delta
419,@user then later he snuggles up to The Royals ...,@user then later he snuggles up to The Royals ...,#Irony,Ironic,0.752852,0.246347,-0.506505
177,I am so ready for Monday. 🙆 #sarcasm,I am so ready for Monday. 🙆,#sarcasm,Ironic,0.830733,0.409149,-0.421584
707,@user @user Talking to yourself again Mo #Irony,@user @user Talking to yourself again Mo,#Irony,Non-ironic,0.631789,0.227166,-0.404623
626,Laptop speakers are too quiet for music & simp...,Laptop speakers are too quiet for music & simp...,#Irony,Ironic,0.574306,0.196353,-0.377953
615,Driving into town on very well cleaned country...,Driving into town on very well cleaned country...,#Irony,Ironic,0.776150,0.436415,-0.339735
190,@user Way to lay out yesterday and put it all ...,@user Way to lay out yesterday and put it all ...,#Sarcasm,Ironic,0.666163,0.388400,-0.277763
239,Everyone wants the truth but nobody wants to b...,Everyone wants the truth but nobody wants to b...,#irony,Ironic,0.532832,0.266976,-0.265855
50,Rangers league game with Alloa moved because o...,Rangers league game with Alloa moved because o...,#irony,Ironic,0.565186,0.308557,-0.256629
22,Pretty excited about how you gave up on me. Fi...,Pretty excited about how you gave up on me. Fi...,#sarcasm,Ironic,0.508401,0.282631,-0.225770
580,Did you know iOS 8 features custom keyboards? ...,Did you know iOS 8 features custom keyboards?,#sarcasm,Ironic,0.684184,0.476537,-0.207647


In [22]:
def prediction_label(scores):
    return scores.ge(0.5).map({
        True: "Ironisch",
        False: "Nicht ironisch",
    })


transitions = single_tag.assign(
    transition=(
        prediction_label(single_tag["score_before"])
        + " → "
        + prediction_label(single_tag["score_after"])
    )
)

counts = (
    transitions.groupby(["reference_label", "marker", "transition"])
    .size()
    .reset_index(name="count")
)

counts["n"] = counts.groupby(["reference_label", "marker"])["count"].transform("sum")
counts["percent"] = 100 * counts["count"] / counts["n"]
counts["marker_label"] = counts["marker"] + " (n=" + counts["n"].astype(str) + ")"

fig = px.bar(
    counts,
    x="marker_label",
    y="percent",
    color="transition",
    facet_col="reference_label",
    barmode="stack",
    hover_data=["count", "n"],
    labels={
        "marker_label": "Entfernter Hashtag",
        "percent": "Anteil der Tweets (%)",
        "transition": "Vorhersage vor → nach",
        "reference_label": "Referenzlabel",
    },
    title="Wie ändert sich die Vorhersage nach dem Entfernen eines Hashtags?",
)
fig.update_xaxes(matches=None)
fig.update_yaxes(range=[0, 100])
fig.show()

In [23]:
hashtag = re.compile(r"(?<!\w)#\w+")


def remove_first_other_tag(text):
    for match in hashtag.finditer(text):
        if not explicit_tag.fullmatch(match.group()):
            edited = text[:match.start()] + text[match.end():]
            return edited.strip(), match.group()
    return text, ""


control = single_tag.copy()
other_edits = control["text"].map(remove_first_other_tag)
control["without_other_tag"] = other_edits.map(lambda edit: edit[0])
control["other_tag"] = other_edits.map(lambda edit: edit[1])
control = control.loc[control["other_tag"].ne("")].copy()

if control.empty:
    print("Keine Tweets mit beiden Hashtag-Arten gefunden.")
else:
    control["delta_other"] = (
        irony_scores(control["without_other_tag"].tolist())
        - control["score_before"]
    )

    fig = px.scatter(
        control,
        x="delta_other",
        y="score_delta",
        color="reference_label",
        opacity=0.65,
        hover_data=[
            "text", "edited_text", "without_other_tag", "removed_tags", "other_tag"
        ],
        labels={
            "delta_other": "Score-Änderung ohne sonstigen Hashtag",
            "score_delta": "Score-Änderung ohne Ironie-Hashtag",
            "reference_label": "Referenzlabel",
        },
        title=f"Welcher Hashtag verändert den Ironie-Score stärker? (n={len(control)})",
    )
    fig.add_shape(
        type="line", x0=-1, y0=-1, x1=1, y1=1,
        line=dict(color="gray", dash="dash"),
    )
    fig.add_hline(y=0, line_dash="dot")
    fig.add_vline(x=0, line_dash="dot")
    fig.update_xaxes(range=[-1, 1])
    fig.update_yaxes(range=[-1, 1], scaleanchor="x", scaleratio=1)
    fig.show()

In [24]:
negation = re.compile(r"\b(?:not|never|no)\b|n['’]t\b", re.IGNORECASE)

heat_data = single_tag.loc[single_tag["score_before"].ge(0.5)].copy()
heat_data["feature"] = heat_data["edited_text"].str.contains(negation).map({
    True: "Negationsmarker gefunden",
    False: "Kein Negationsmarker gefunden",
})
heat_data["flip_to_non_ironic"] = heat_data["score_after"].lt(0.5)

summary = heat_data.groupby(["feature", "marker"]).agg(
    flips=("flip_to_non_ironic", "sum"),
    n=("flip_to_non_ironic", "size"),
)
summary["percent"] = 100 * summary["flips"] / summary["n"]
summary["annotation"] = (
    summary["flips"].astype(str) + "/" + summary["n"].astype(str)
)

rates = summary["percent"].unstack()
annotations = summary["annotation"].unstack().reindex_like(rates).fillna("")

fig = px.imshow(
    rates,
    zmin=0,
    zmax=100,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Entfernter Hashtag",
        "y": "Merkmal im verbleibenden Text",
        "color": "Klassenwechsel (%)",
    },
    title="Wechsel von ironisch zu nicht ironisch nach Textmerkmal",
)
fig.update_traces(text=annotations.to_numpy(), texttemplate="%{text}")
fig.show()

In [25]:
emoji_test = tweets[["text", "reference_label", "score_before"]].copy()

emoji_test["removed_emojis"] = emoji_test["text"].map(
    lambda text: " ".join(item["emoji"] for item in emoji.emoji_list(text))
)

# Nur Tweets erneut bewerten, die tatsächlich Emojis enthalten.
emoji_test = emoji_test.loc[emoji_test["removed_emojis"].ne("")].copy()

emoji_test["without_emojis"] = emoji_test["text"].map(
    lambda text: emoji.replace_emoji(text, replace="").strip()
)

emoji_test["score_after_emoji"] = irony_scores(
    emoji_test["without_emojis"].tolist()
)
emoji_test["delta_emoji"] = (
    emoji_test["score_after_emoji"] - emoji_test["score_before"]
)

before_ironic = emoji_test["score_before"].ge(0.5)
after_ironic = emoji_test["score_after_emoji"].ge(0.5)

emoji_test["transition"] = "Kein Klassenwechsel"
emoji_test.loc[before_ironic & ~after_ironic, "transition"] = (
    "Ironisch → Nicht ironisch"
)
emoji_test.loc[~before_ironic & after_ironic, "transition"] = (
    "Nicht ironisch → Ironisch"
)

emoji_test["class_flip"] = before_ironic.ne(after_ironic)

print(f"{len(emoji_test)} von {len(tweets)} Tweets enthalten Emojis.")

emoji_test.groupby("reference_label").agg(
    tweets_mit_emojis=("text", "size"),
    median_score_aenderung=("delta_emoji", "median"),
    klassenwechsel=("class_flip", "sum"),
)

88 von 784 Tweets enthalten Emojis.


,tweets_mit_emojis,median_score_aenderung,klassenwechsel
reference_label,,,
Ironic,33,0.070080,6
Non-ironic,55,0.029399,8


In [26]:
if emoji_test.empty:
    print("Im Testsplit wurden keine Emojis gefunden.")
else:
    fig = px.scatter(
        emoji_test,
        x="score_before",
        y="score_after_emoji",
        color="reference_label",
        symbol="class_flip",
        opacity=0.65,
        category_orders={
            "reference_label": ["Ironic", "Non-ironic"],
            "class_flip": [False, True],
        },
        hover_data={
            "text": True,
            "without_emojis": True,
            "removed_emojis": True,
            "delta_emoji": ":.3f",
        },
        labels={
            "score_before": "Ironie-Score · Original",
            "score_after_emoji": "Ironie-Score · ohne Emojis",
            "reference_label": "Referenzlabel",
            "class_flip": "Klassenwechsel",
            "text": "Originaltext",
            "without_emojis": "Text ohne Emojis",
            "removed_emojis": "Entfernte Emojis",
            "delta_emoji": "Score-Änderung",
        },
        template="plotly_white",
    )

    fig.add_shape(
        type="line",
        x0=0, y0=0, x1=1, y1=1,
        line=dict(color="gray", dash="dash"),
    )
    fig.add_vline(x=0.5, line_dash="dot")
    fig.add_hline(y=0.5, line_dash="dot")
    fig.update_xaxes(range=[0, 1], tick0=0, dtick=0.2, constrain="domain")
    fig.update_yaxes(
        range=[0, 1],
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    )
    legend_labels = {
        "Ironic, False": "Ironisch",
        "Ironic, True": "Ironisch · Wechsel",
        "Non-ironic, False": "Nicht ironisch",
        "Non-ironic, True": "Nicht ironisch · Wechsel",
    }
    fig.for_each_trace(lambda trace: trace.update(name=legend_labels[trace.name]))
    fig.update_xaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_yaxes(dtick=0.2, gridcolor="#E7EDF4")
    fig.update_layout(
        width=600,
        modebar_remove=["toImage", "select2d", "lasso2d", "zoomIn2d",
                        "zoomOut2d", "autoScale2d"],
        height=640,
        margin=dict(l=61, r=19, t=66, b=54),
        title=dict(
            text=f"Emoji-Entfernung · {len(emoji_test)} Tweets",
            x=0,
            xref="paper",
            xanchor="left",
            y=0.985,
            yanchor="top",
            font=dict(size=21),
        ),
        legend=dict(
            orientation="h",
            x=0,
            xanchor="left",
            y=1.012,
            yanchor="bottom",
            title_text="",
            font=dict(size=11),
        ),
    )
    fig.show()